Incorporate into function to run all with variables: channel list, chromophore, age, location

In [12]:
%matplotlib inline

%reset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import skfda
import sys

from matplotlib import cm
from scipy.integrate import simpson
from skfda.preprocessing.dim_reduction import FPCA
from skfda.representation.basis import BSplineBasis
from statsmodels.stats.multitest import multipletests

sys.path.append("/Users/sambe/Documents/Python_Projects/FDA/functions")
import fNIRSData
import funcChangePoint
import auxFuncChPt

import importlib
importlib.reload(fNIRSData)
importlib.reload(funcChangePoint)
importlib.reload(auxFuncChPt)

Nothing done.


<module 'auxFuncChPt' from '/Users/sambe/Documents/Python_Projects/FDA/functions/auxFuncChPt.py'>

Define input file/output filename (if you want to save) and the channels you wish to analyse (i,e. ROI channels)
<br> If adding more channels, the blocks of code which need to be included in a for loop are indicated below in the notebook.

In [13]:
# User defined params
location = 'GM'
agePoint = 5
numChrom = 2
firstTrial = 1
lastTrial = 25
cptSignificance = 0.01
# cmultiple comparison correction methods - e.g. 'b' = Bonferroni
withinChanCorrectMethod = 'h' #'h' = Bonferroni-Holm
firstSamp = -20
lastSamp = 182
channels = 'ALL' #'ROI' for ROI channels base on previous analysis; 'ALL' for all array channels
useLongRun = False #see autocorrelation test and ACF figures below!
lrCovLag = 0 #set to 0 to calculate h based on dataset size
useWildBinSeg = True

#for indexing later on
firstFunc = firstTrial-1
lastFunc = lastTrial-1

In [14]:
# Dependent on other user-defined params (above)
if numChrom == 1:
    chromName = 'HbO'
elif numChrom == 2:
    chromName = 'HbR'
else:
    chromName = 'HbT'

dataFile = '/Users/sambe/Documents/Python_Projects/FDA/dataRaw/homer2'+str(location)+str(agePoint)+'moDataFrame.pkl'
cptFilename = '/Users/sambe/Documents/Python_Projects/FDA/data/grpChangePointDFs/ChPt'+str(location)+str(agePoint)+'mo'+str(chromName)+'FPCA.pkl'
volFilename = '/Users/sambe/Documents/Python_Projects/FDA/data/grpVolRatioDFs/VolRatio'+str(location)+str(agePoint)+'mo'+str(chromName)+'FPCA.pkl'

if channels.upper() == 'ROI':
    #(CPA) ROI channels
    channels = fNIRSData.getROI(agePoint, location)
elif channels.upper() == 'ALL':
    with open(dataFile, 'rb') as f:
        dfFull = pickle.load(f)
    channels = (np.unique(dfFull["Channel"].values)).tolist()
else:
    print("Using ALL array channels")
    with open(dataFile, 'rb') as f:
        dfFull = pickle.load(f)
    channels = (np.unique(dfFull["Channel"].values)).tolist()

In [15]:
columnList = ['Channel',
'Chromophore',
'Recursion Level',
'First Trial',
'Last Trial',
'Change Point',
'Max. T_N',
'p-value']

dfChangePoints = pd.DataFrame(columns = columnList)

gridPoints = np.arange(firstSamp, lastSamp, 1) # for func. data

allPValues = []  # initialise storage all p-values for across-channel correction
globalIndex = 0 #for dfChangePoints indexing

If working with multiple channels, group the following 5 blocks of code together and run inside a for loop, i.e.:
<br> for numChan in channels: 

In [16]:
for numChan in channels:

    dataMatrix = fNIRSData.groupDataMatrix(dataFile, numChan, numChrom, firstTrial, lastTrial)
    fd = skfda.FDataGrid(data_matrix=dataMatrix, grid_points=gridPoints)
    dHat = auxFuncChPt.cumulPercVariance(fd)

    chanMat, pValues = auxFuncChPt.findCPtsFPCA(dataMatrix, firstFunc, lastFunc, dHat = dHat, alphaVal = cptSignificance, h = lrCovLag, longRun = useLongRun, wildBinSeg = useWildBinSeg)
    chanMat = chanMat.astype('float32')
    #chanMat = np.delete(chanMat, (0), axis=0) #deletes a dummy row in the matrix which is added during it's creation

    #within channel correction (multiple tests due to binseg approach)
    pAdjWithin = multipletests(pValues, alpha = cptSignificance, method=withinChanCorrectMethod)[1]  # within-channel multiple comparison correction (accounts for BinSeg)

    # initialise temporary channel dataframe rows
    tempRows = []
    #populate temp. dataframe with change point results
    for i in range(chanMat.shape[0]):
        tempRows.append({
            'Channel':numChan,
            'Chromophore':numChrom,
            'Recursion Level': (chanMat[i][4]),
            'First Trial':(chanMat[i][0]+1),
            'Last Trial': (chanMat[i][1]+1),
            'Change Point': (chanMat[i][2]+1),
            'Max. T_N': (chanMat[i][3]),
            'p-value': (chanMat[i][5]) 
        })
    # create temp. dataframe
    tempDf = pd.DataFrame(tempRows)
    if not tempDf.empty:
        tempDf.index = range(globalIndex, globalIndex + len(tempDf))

    filteredTempDf = auxFuncChPt.filterSignificantChanges(tempDf, cptSignificance, firstTrial, lastTrial)

    if not filteredTempDf.empty:

        #add filtered results to main dataframe
        dfChangePoints = pd.concat([dfChangePoints, filteredTempDf])

        #update global index
        globalIndex = dfChangePoints.index.max() + 1 if not dfChangePoints.empty else 0
        
    print('Progress: '+str(round((channels.index(numChan)+1) / len(channels) * 100, 1))+'%') 

Progress: 2.9%
Progress: 5.9%
Progress: 8.8%
Progress: 11.8%
Progress: 14.7%
Progress: 17.6%
Progress: 20.6%
Progress: 23.5%
Progress: 26.5%
Progress: 29.4%
Progress: 32.4%
Progress: 35.3%
Progress: 38.2%
Progress: 41.2%
Progress: 44.1%
Progress: 47.1%
Progress: 50.0%
Progress: 52.9%
Progress: 55.9%
Progress: 58.8%
Progress: 61.8%
Progress: 64.7%
Progress: 67.6%
Progress: 70.6%
Progress: 73.5%
Progress: 76.5%
Progress: 79.4%
Progress: 82.4%
Progress: 85.3%
Progress: 88.2%
Progress: 91.2%
Progress: 94.1%
Progress: 97.1%
Progress: 100.0%


In [17]:
#save the ChPt dataframe
with open(cptFilename, 'wb') as f:
    pickle.dump(dfChangePoints, f)

In [18]:
"""

columnList = ['Channel',
'Final Change Point',
'Vol No Habituation',
'Vol Habituation',
'Ratio of Volumes']

dfVolumes = pd.DataFrame(columns = columnList)

#get list of only those channels in which a CPt has been detected (otherwise a volume calculation is pointless!)
cptChannels =np.unique(dfChangePoints["Channel"].values)

for numChan in cptChannels:

    dfChan = (dfChangePoints[dfChangePoints['Channel'] == numChan]).sort_values(by=['Change Point'])

    dataMatrix = fNIRSData.groupDataMatrix(dataFile,
                                        numChan,
                                        numChrom,
                                        firstTrial,
                                        lastTrial)

    changeColumn = (dfChan.get('Change Point').tolist())
    changeColumn = [int(x) for x in changeColumn]
    changeColumn.append(lastTrial + 1)
    
    iBlock = 0
    firstBlockTrial = 1
    blockMeans = np.empty([len(changeColumn), (abs(firstSamp)+abs(lastSamp))])

    for point in changeColumn:
        lastBlockTrial = point-1 #-1 because dataMatrix rows go from 0->(numTrials-1)
        blockMatrix = dataMatrix[firstBlockTrial-1:lastBlockTrial-1, ]
        blockMeans[iBlock] = np.mean(blockMatrix, axis = 0)
        firstBlockTrial = point
        iBlock = iBlock + 1

    changeColumn.insert(0,1)

    volNoHab = simpson(blockMeans[0, 21:], dx = 1) * lastTrial
    volHab = 0
    for i in range(0, len(changeColumn)-1):
        volTrial = (simpson(blockMeans[i, 21:], dx = 1)) * (changeColumn[i+1] - changeColumn[i])
        volHab = volHab + volTrial

    ratioHab = (volHab / volNoHab)

    #save variable for plotting/demo
    if numChan == 3:
        blockMeansPlot = blockMeans


    dfVolumes = pd.concat([dfVolumes, pd.DataFrame.from_records([{
            'Channel':numChan,
            'Final Change Point':changeColumn[-2],
            'Vol No Habituation':abs(volNoHab),
            'Vol Habituation': abs(volHab),
            'Ratio of Volumes': ratioHab
            }])])
"""

'\n\ncolumnList = [\'Channel\',\n\'Final Change Point\',\n\'Vol No Habituation\',\n\'Vol Habituation\',\n\'Ratio of Volumes\']\n\ndfVolumes = pd.DataFrame(columns = columnList)\n\n#get list of only those channels in which a CPt has been detected (otherwise a volume calculation is pointless!)\ncptChannels =np.unique(dfChangePoints["Channel"].values)\n\nfor numChan in cptChannels:\n\n    dfChan = (dfChangePoints[dfChangePoints[\'Channel\'] == numChan]).sort_values(by=[\'Change Point\'])\n\n    dataMatrix = fNIRSData.groupDataMatrix(dataFile,\n                                        numChan,\n                                        numChrom,\n                                        firstTrial,\n                                        lastTrial)\n\n    changeColumn = (dfChan.get(\'Change Point\').tolist())\n    changeColumn = [int(x) for x in changeColumn]\n    changeColumn.append(lastTrial + 1)\n    \n    iBlock = 0\n    firstBlockTrial = 1\n    blockMeans = np.empty([len(changeColumn), (

In [19]:
"""
#save the Vol dataframe
with open(volFilename, 'wb') as f:
    pickle.dump(dfVolumes, f)
    """

"\n#save the Vol dataframe\nwith open(volFilename, 'wb') as f:\n    pickle.dump(dfVolumes, f)\n    "

In [20]:
"""

#plotting here works well on 12mo GM channel 3
trialFrame = pd.DataFrame(data=blockMeansPlot)
trialFrame = trialFrame.T
trialFrame.columns = trialFrame.columns + 1

trialFrame.plot(figsize=[4,4], 
                cmap = cm.viridis, 
                xlabel = 'Time(ds)', 
                ylabel = 'Concentration change ($\mu$M)')
                """

"\n\n#plotting here works well on 12mo GM channel 3\ntrialFrame = pd.DataFrame(data=blockMeansPlot)\ntrialFrame = trialFrame.T\ntrialFrame.columns = trialFrame.columns + 1\n\ntrialFrame.plot(figsize=[4,4], \n                cmap = cm.viridis, \n                xlabel = 'Time(ds)', \n                ylabel = 'Concentration change ($\\mu$M)')\n                "

In [21]:
"""

plt.style.use('_mpl-gallery')

# Make data
Y = np.arange(0, dataMatrix.shape[0])
X = np.arange(0, dataMatrix.shape[1])

X, Y = np.meshgrid(X, Y)

fig = plt.figure(figsize=[8,8]) # !! Change figure size here!! 
ax = fig.add_subplot( projection="3d")
ax.set_zlabel('Concentration change ($\mu$M)') #can't set z label using plt
ax.set_box_aspect(aspect=None, zoom=0.95) #so z label can be seen

#using blues:
#ax.plot_surface(Y, X, dataMatrix, cmap=cm.Blues, lw=0.5, rstride=1, cstride=1)

#colourblind friendly:
ax.plot_surface(Y, X, dataMatrix, cmap = cm.viridis, lw=0.5, rstride=1, cstride=1)

plt.title('Mean haemodynamic responses in channel '+str(numChan)+' across all '+str(agePoint)+'mo participants')
plt.xlabel('Trial') 
plt.ylabel('Time (ds)')

plt.show()
"""

'\n\nplt.style.use(\'_mpl-gallery\')\n\n# Make data\nY = np.arange(0, dataMatrix.shape[0])\nX = np.arange(0, dataMatrix.shape[1])\n\nX, Y = np.meshgrid(X, Y)\n\nfig = plt.figure(figsize=[8,8]) # !! Change figure size here!! \nax = fig.add_subplot( projection="3d")\nax.set_zlabel(\'Concentration change ($\\mu$M)\') #can\'t set z label using plt\nax.set_box_aspect(aspect=None, zoom=0.95) #so z label can be seen\n\n#using blues:\n#ax.plot_surface(Y, X, dataMatrix, cmap=cm.Blues, lw=0.5, rstride=1, cstride=1)\n\n#colourblind friendly:\nax.plot_surface(Y, X, dataMatrix, cmap = cm.viridis, lw=0.5, rstride=1, cstride=1)\n\nplt.title(\'Mean haemodynamic responses in channel \'+str(numChan)+\' across all \'+str(agePoint)+\'mo participants\')\nplt.xlabel(\'Trial\') \nplt.ylabel(\'Time (ds)\')\n\nplt.show()\n'

In [22]:
"""

blockSurfGrid = np.empty([dataMatrix.shape[0]*1000, dataMatrix.shape[1]])
for i in range(0, dataMatrix.shape[0]*1000):
    posTemp = ([j for j in range(len(changeColumn)) if changeColumn[j] <= (i/1000)+1])
    blockSurfGrid[i] = blockMeansPlot[max(posTemp)]
    
plt.style.use('_mpl-gallery')

# Make data
Y = np.arange(0, dataMatrix.shape[0], 0.001)
X = np.arange(0, dataMatrix.shape[1])

X, Y = np.meshgrid(X, Y)

fig = plt.figure(figsize=[8,8]) # !! Change figure size here!! 
ax = fig.add_subplot( projection="3d")
ax.grid(b = True, which = 'major')
ax.set_zlabel('Concentration change ($\mu$M)') #can't set z label using plt
ax.set_box_aspect(aspect=None, zoom=0.95) #so z label can be seen

#using blues:
#ax.plot_surface(Y, X, dataMatrix, cmap=cm.Blues, lw=0.5, rstride=1, cstride=1)

#colourblind friendly:
ax.plot_surface(Y, X, blockSurfGrid, cmap = cm.viridis, lw=0.5, rstride=1000, cstride=1)

plt.grid(visible = True, which = 'major')
plt.title('Mean haemodynamic responses between change points in channel '+str(numChan)+' across all '+str(agePoint)+'mo participants')
plt.xlabel('Trial') 
plt.ylabel('Time (ds)')

plt.show()
"""

'\n\nblockSurfGrid = np.empty([dataMatrix.shape[0]*1000, dataMatrix.shape[1]])\nfor i in range(0, dataMatrix.shape[0]*1000):\n    posTemp = ([j for j in range(len(changeColumn)) if changeColumn[j] <= (i/1000)+1])\n    blockSurfGrid[i] = blockMeansPlot[max(posTemp)]\n    \nplt.style.use(\'_mpl-gallery\')\n\n# Make data\nY = np.arange(0, dataMatrix.shape[0], 0.001)\nX = np.arange(0, dataMatrix.shape[1])\n\nX, Y = np.meshgrid(X, Y)\n\nfig = plt.figure(figsize=[8,8]) # !! Change figure size here!! \nax = fig.add_subplot( projection="3d")\nax.grid(b = True, which = \'major\')\nax.set_zlabel(\'Concentration change ($\\mu$M)\') #can\'t set z label using plt\nax.set_box_aspect(aspect=None, zoom=0.95) #so z label can be seen\n\n#using blues:\n#ax.plot_surface(Y, X, dataMatrix, cmap=cm.Blues, lw=0.5, rstride=1, cstride=1)\n\n#colourblind friendly:\nax.plot_surface(Y, X, blockSurfGrid, cmap = cm.viridis, lw=0.5, rstride=1000, cstride=1)\n\nplt.grid(visible = True, which = \'major\')\nplt.title(\'